# Godot 生态仓库 · 完整数据表(红绿色阶)

数据来自本仓库 `godot.db`(5 个 topic、`star ≥ 1000`、最近 3 个月有推送,去重后 **51 个仓库**,全部列出)。

维度:**full_name / url / stars / forks / open_issues / 月均新增 star**。

> 月均新增 star = `stars ÷ 从创建到最后更新的月数`,衡量仓库历史上的平均涨星速度。
> 数值列按列做**红绿色阶**(RdYlGn:同列内越低越红、越高越绿)。
> 着色用每个单元格的 inline 样式,**GitHub 在线查看即可看到颜色**。

In [1]:
import sqlite3
import pandas as pd
import matplotlib
from IPython.display import HTML

conn = sqlite3.connect('godot.db')
df = pd.read_sql_query(
    'SELECT full_name, url, stars, forks, open_issues, created_at, updated_at FROM repos',
    conn,
)
conn.close()

# 月均新增 star = stars / 从创建到最后更新的月数(至少按 1 个月计,避免除零)
created = pd.to_datetime(df['created_at'], utc=True)
updated = pd.to_datetime(df['updated_at'], utc=True)
active_months = ((updated - created).dt.days / 30.44).clip(lower=1.0)
df['stars_per_month'] = (df['stars'] / active_months).round(1)
df = (df.drop(columns=['created_at', 'updated_at'])
        .sort_values('stars', ascending=False)
        .reset_index(drop=True))
print(f'{len(df)} 个仓库')
df.head()

51 个仓库


,full_name,url,stars,forks,open_issues,stars_per_month
0,godotengine/godot,https://github.com/godotengine/godot,112899,25726,18480,755.1
1,Donchitos/Claude-Code-Game-Studios,https://github.com/Donchitos/Claude-Code-Game-...,22010,3181,44,5153.7
2,heroiclabs/nakama,https://github.com/heroiclabs/nakama,12778,1430,124,112.9
3,godotengine/awesome-godot,https://github.com/godotengine/awesome-godot,10210,549,53,76.0
4,Orama-Interactive/Pixelorama,https://github.com/Orama-Interactive/Pixelorama,9771,516,83,119.0


In [2]:
# 数值列统一红绿色阶(RdYlGn):同列内归一化,低=红、高=绿
NUM_COLS = ['stars', 'forks', 'open_issues', 'stars_per_month']
CMAP = matplotlib.colormaps['RdYlGn']
ALPHA = 0.55

def _cell_bg(value, vmin, vmax):
    t = 0.0 if vmax == vmin else (value - vmin) / (vmax - vmin)
    r, g, b, _ = CMAP(t)
    return f'rgba({int(r*255)},{int(g*255)},{int(b*255)},{ALPHA})'

def render_table(frame, caption):
    bounds = {c: (frame[c].min(), frame[c].max()) for c in NUM_COLS if c in frame.columns}
    head_style = ('background-color:#222;color:#fff;padding:6px 10px;'
                  'text-align:left;position:sticky;top:0')
    html = [f'<table style="border-collapse:collapse;font-family:system-ui,Arial,sans-serif;'
            f'font-size:13px"><caption style="font-size:16px;font-weight:bold;'
            f'padding:8px;text-align:left">{caption}</caption>']
    html.append('<tr><th style="' + head_style + '">#</th>' + ''.join(
        f'<th style="{head_style}">{col}</th>' for col in frame.columns) + '</tr>')
    for rank, (_, row) in enumerate(frame.iterrows(), start=1):
        base = 'padding:4px 10px;border-bottom:1px solid #eee'
        cells_html = [f'<td style="{base};color:#888">{rank}</td>']
        for col in frame.columns:
            v = row[col]
            if col == 'full_name':
                cells_html.append(f'<td style="{base};font-weight:bold">{v}</td>')
            elif col == 'url':
                cells_html.append(
                    f'<td style="{base}"><a href="{v}" target="_blank">link</a></td>')
            elif col in NUM_COLS:
                vmin, vmax = bounds[col]
                bg = _cell_bg(v, vmin, vmax)
                txt = f'{v:,.1f}' if col == 'stars_per_month' else f'{int(v):,}'
                cells_html.append(
                    f'<td style="{base};background-color:{bg};text-align:right">{txt}</td>')
            else:
                cells_html.append(f'<td style="{base}">{v}</td>')
        html.append('<tr>' + ''.join(cells_html) + '</tr>')
    html.append('</table>')
    return HTML('\n'.join(html))

# 完整数据表:全部 51 个仓库,按 stars 降序
render_table(df, f'Godot 生态完整数据表 — 共 {len(df)} 个仓库,数值列红绿色阶')

#,full_name,url,stars,forks,open_issues,stars_per_month
1,godotengine/godot,link,"112,899","25,726","18,480",755.1
2,Donchitos/Claude-Code-Game-Studios,link,"22,010","3,181",44,"5,153.7"
3,heroiclabs/nakama,link,"12,778","1,430",124,112.9
4,godotengine/awesome-godot,link,"10,210",549,53,76.0
5,Orama-Interactive/Pixelorama,link,"9,771",516,83,119.0
6,0xFA11/MultiplayerNetworkingResources,link,"8,556",534,2,84.8
7,Redot-Engine/redot-engine,link,"5,880",306,61,56.4
8,dialogic-godot/dialogic,link,"5,726",335,168,79.0
9,RodZill4/material-maker,link,"5,554",352,311,58.5
10,godotengine/godot-docs,link,"5,430","3,686","1,077",43.0
